# ONNX → DXNN 컴파일 (Colab)

학습 노트북이 만든 분류 ONNX(`resnet18_v1.onnx` 또는 `mobilenet_v2_v1.onnx`)를 Colab에서 DX-COM 2.5.0으로 INT8 `.dxnn`으로 컴파일한다.

Calibration은 `calib_100.npy` 한 파일이다. shape `(100, 3, 64, 64)`, dtype `uint8`, 값 `0–255`. 클래스 이름은 없다. DX-COM은 정답을 쓰지 않고 이 텐서만 모델에 넣어 activation 범위를 잰다.

`dx-com` wheel은 **x86_64 Linux** 전용이다. Raspberry Pi에서는 설치되지 않는다. Colab에는 NPU가 없으므로 여기서는 **컴파일만** 한다. `.dxnn` 실행은 보드의 `dx_engine`에서 한다.

**준비.** Google Drive `MyDrive/ondevice/`에 아래가 있어야 한다.

| 파일 | 용도 |
|---|---|
| `calib_100.npy` | calibration 100장. `(100, 3, 64, 64)` uint8, 0–255 |
| `runs/<모델>_v1/<모델>_v1.onnx` | 학습 노트북이 저장한 ONNX |

`windows_*.npz`, `features.py`는 이 노트북에서 다시 읽지 않는다.

**실행:** 런타임 → 런타임 유형 변경 → T4 GPU → 위에서부터 실행. CPU만 있어도 컴파일은 된다.

결과(`*.dxnn`, `compiler.log`, `*_summary.html`)는 `MyDrive/ondevice/runs/<모델>_v1/`에 저장된다.

## `calib_100.npy`가 맞아야 하는 값

ONNX 입력 이름은 `images`, shape는 `[1, 3, 64, 64]`이다. DataLoader는 배치 차원을 뺀 `[3, 64, 64]` `float32`를 반환하고 `batch_size`는 1이다. npy의 uint8은 셀에서 `float32`로만 바꾼다. `/255`나 `/127.5 - 1`은 하지 않는다. 그 나눗셈은 ONNX 그래프 안에 있다.

| 확인 | 기대 |
|---|---|
| shape | `(100, 3, 64, 64)` |
| 값 | `0–255` |
| 채널 0 평균 | Mel dB. 128에 몰리지 않음 |
| 채널 1·2 평균 | 시간 변화량. 128 근처 |

클래스 열이 없어도 컴파일은 된다. 100장을 만들 때 명령(휘파람, 혀 차기, 박수, shh)이 포함돼 있어야 그 클래스의 INT8 범위가 잡힌다. 이 노트북은 그 포함 여부를 파일 안에서 다시 검사할 수 없다.

JSON `config`의 `default_loader`는 쓰지 않는다. 그 경로는 사진을 BGR로 읽고 `/255`를 붙인다. `dataloader`와 `config`를 같이 넘기면 `ValueError`가 난다.

In [ ]:
# 1. 설정. 모델을 바꿀 때는 MODEL만 바꾼다.
MODEL = "resnet18"                 # "resnet18" 또는 "mobilenet_v2"
VERSION = "v1"
DRIVE = "/content/drive/MyDrive/ondevice"
WORK = "/content/compile"          # Drive가 아니라 로컬 디스크에서 컴파일
CALIB_NAME = "calib_100.npy"

In [ ]:
# 2. DX-COM 2.5.0. wheel은 x86_64 + glibc 2.31 이상 (Colab은 해당, Raspberry Pi는 아님).
!apt-get install -y -qq libgl1 libglib2.0-0
!pip -q install "dx-com==2.5.0" onnx
import dx_com
print("dx_com", dx_com.__version__)

In [ ]:
# 3. Drive에서 ONNX와 calib_100.npy를 Colab 로컬로 복사
from google.colab import drive
drive.mount("/content/drive")
import os, shutil
os.makedirs(WORK, exist_ok=True)

CALIB_SRC = f"{DRIVE}/{CALIB_NAME}"
if not os.path.exists(CALIB_SRC):
    raise FileNotFoundError(f"Drive에 없음: {CALIB_SRC}")
CALIB_LOCAL = f"{WORK}/{CALIB_NAME}"
shutil.copy(CALIB_SRC, CALIB_LOCAL)

RUN = f"{DRIVE}/runs/{MODEL}_{VERSION}"
ONNX_SRC = f"{RUN}/{MODEL}_{VERSION}.onnx"
if not os.path.exists(ONNX_SRC):
    raise FileNotFoundError(f"ONNX 없음: {ONNX_SRC}. 학습 노트북 export를 먼저 실행한다.")
ONNX_LOCAL = f"{WORK}/{MODEL}_{VERSION}.onnx"
shutil.copy(ONNX_SRC, ONNX_LOCAL)
print("ONNX", ONNX_LOCAL, os.path.getsize(ONNX_LOCAL))
print("calib", CALIB_LOCAL, os.path.getsize(CALIB_LOCAL))

In [ ]:
# 4. calib_100.npy 검사. 클래스 이름은 없고, 텐서 형식만 확인한다.
import numpy as np
raw = np.load(CALIB_LOCAL)
print("raw", raw.shape, raw.dtype, "min", int(raw.min()), "max", int(raw.max()))
if raw.shape != (100, 3, 64, 64):
    raise RuntimeError(f"shape {raw.shape} != (100, 3, 64, 64)")
if raw.dtype != np.uint8:
    raise RuntimeError(f"dtype {raw.dtype} != uint8")
if int(raw.min()) < 0 or int(raw.max()) > 255:
    raise RuntimeError("값이 0–255 밖이다")

feats = raw.astype(np.float32)                 # 범위는 그대로 0–255. /127.5-1 은 그래프 안
print("float32", feats.shape, "mean", float(feats.mean()))
print("채널별 mean (0=dB, 1·2=변화량, 변화 없음은 128 근처)")
for ch in range(3):
    sl = feats[:, ch]
    print(f"  ch{ch}  min {sl.min():7.2f}  mean {sl.mean():7.2f}  max {sl.max():7.2f}")
if feats[:, 1].mean() < 110 or feats[:, 2].mean() < 110:
    raise RuntimeError("채널 1·2 평균이 128에서 멀리 있다. spectrogram feature가 아닐 수 있다.")
if feats.mean() < 2:
    raise RuntimeError("값이 0–1로 보인다. /255 를 적용한 배열이다.")

In [ ]:
# 5. DX-COM DataLoader. 배치는 1. 반환 shape는 [3, 64, 64]. 레이블은 없다.
import torch
from torch.utils.data import Dataset, DataLoader

class CalibSet(Dataset):
    def __len__(self):
        return len(feats)
    def __getitem__(self, k):
        return torch.from_numpy(feats[k])          # float32 [3, 64, 64], 0–255

g = torch.Generator().manual_seed(0)
loader = DataLoader(CalibSet(), batch_size=1, shuffle=True, generator=g)
batch = next(iter(loader))
print("batch", tuple(batch.shape), batch.dtype, "min", float(batch.min()), "max", float(batch.max()))
if tuple(batch.shape) != (1, 3, 64, 64):
    raise RuntimeError(f"batch shape {tuple(batch.shape)} != (1, 3, 64, 64)")

In [ ]:
# 6. ONNX가 학습 노트북 export와 같은지 확인한 뒤에만 컴파일한다.
import onnx
model = onnx.load(ONNX_LOCAL)
opset = model.opset_import[0].version
inp = model.graph.input[0]
shape = [d.dim_value for d in inp.type.tensor_type.shape.dim]
out_name = model.graph.output[0].name
print(f"opset {opset}  input {inp.name} {shape}  output {out_name}")
if not (11 <= opset <= 21):
    raise RuntimeError(f"opset {opset}는 DX-COM 범위(11–21) 밖")
if inp.name != "images" or shape != [1, 3, 64, 64]:
    raise RuntimeError("입력이 images [1, 3, 64, 64]가 아님. 학습 노트북 export를 확인한다.")
if any(d <= 0 for d in shape):
    raise RuntimeError(f"동적 shape {shape}. batch=1, 모든 차원은 고정이어야 한다.")
onnx.checker.check_model(model)
print("ONNX checker PASS")

In [ ]:
# 7. 컴파일. config.json은 넘기지 않는다. calibration_num은 npy 장수(100).
# PPU는 검출 모델용이라 분류에는 넣지 않는다.
import glob
print("CUDA", torch.cuda.is_available(), "calibration", len(feats))
dx_com.compile(
    model=ONNX_LOCAL,
    output_dir=WORK,
    dataloader=loader,
    calibration_method="ema",
    calibration_num=len(feats),
    opt_level=1,
    gen_log=True,
    export_html=True,
    use_gpu=torch.cuda.is_available(),
    verbose=True,
)

made = sorted(glob.glob(f"{WORK}/*"))
for p in made:
    print(os.path.basename(p), os.path.getsize(p))
dxnn = f"{WORK}/{MODEL}_{VERSION}.dxnn"
if not os.path.exists(dxnn):
    cands = glob.glob(f"{WORK}/*.dxnn")
    if len(cands) != 1:
        raise FileNotFoundError(f".dxnn을 찾지 못함: {cands}")
    dxnn = cands[0]
os.makedirs(RUN, exist_ok=True)
for p in made:
    if os.path.isfile(p):
        shutil.copy(p, f"{RUN}/{os.path.basename(p)}")
print("Drive 복사 완료:", RUN)

## 컴파일 다음

`compiler.log`에서 두 곳을 본다.

- `Added nodes`: 여기 적힌 연산은 NPU 그래프에 들어갔다. 보드 전처리에서 또 하면 두 번 적용된다.
- `INPUT PREPROCESSING GUIDANCE`: 보드가 `.dxnn`에 넣을 dtype과 layout. 이 모델은 입력을 0–255로 받고 그래프 안에서 `x / 127.5 - 1`을 하므로, 안내에 다른 말이 없으면 보드에는 `features.py`의 uint8 `[1, 3, 64, 64]`를 그대로 넣는다.

`*_summary.html`은 컴파일 요약이다. Colab에서 `.dxnn`을 실행해 ONNX와 숫자를 비교할 수는 없다. 그 비교는 보드 NPU에서 `dx_engine`으로 한다.